# 01 – Orchestration in Python (Airflow & co.)
**Master's course · 60 min** · companion of `01_orchestration.md`

| Part | Time | Content |
|---|---|---|
| A | 3 min | Why a naive pipeline breaks |
| B | 6 min | Build a mini-orchestrator (DAG, parallelism, retries, failure propagation) |
| C | 3 min | Idempotency & backfill |
| D | 5 min | A real Airflow DAG: write, parse, run |
| E | – | Exercises |

Parts A–C need only the Python standard library. Part D needs Apache Airflow (optional install cell provided).

## Part A – The naive pipeline
Three "scripts" glued in order, one of them calls an unreliable API.

In [ ]:
import random, time

random.seed(1)

def extract():
    if random.random() < 0.6:                 # transient failure (network, rate limit...)
        raise ConnectionError("API timeout")
    return [1, 2, 3, 4, 5]

def transform(rows):
    return [r * 10 for r in rows]

def load(rows):
    print("loaded", rows)

# the "cron" way
try:
    load(transform(extract()))
except Exception as e:
    print("Pipeline crashed:", repr(e), "-> nobody retries, nobody is notified, no history.")

**Discussion (1 min):** list everything the script above is missing (retries, state, logs, dependencies, parallelism, alerting, backfill...). The rest of the notebook adds these one by one.

## Part B – Build a mini-orchestrator
Core ideas implemented with the standard library only:
* **DAG** = tasks + upstream dependencies, solved by `graphlib.TopologicalSorter`
* **Executor** = a thread pool running ready tasks in parallel
* **State machine** per task: `pending → running → success | failed | upstream_failed`
* **Retries** with exponential backoff
* **Results passing** (a tiny "XCom")

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from graphlib import TopologicalSorter
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from datetime import datetime
from typing import Callable

class State(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"
    UPSTREAM_FAILED = "upstream_failed"

@dataclass
class Task:
    name: str
    fn: Callable
    upstream: list = field(default_factory=list)
    retries: int = 0
    retry_delay: float = 0.2          # seconds, doubled at each attempt

class MiniDAG:
    def __init__(self, dag_id: str):
        self.dag_id = dag_id
        self.tasks: dict[str, Task] = {}

    # decorator to register a task
    def task(self, after=(), retries=0, retry_delay=0.2):
        def deco(fn):
            t = Task(fn.__name__, fn, [a.name for a in after], retries, retry_delay)
            self.tasks[t.name] = t
            return t
        return deco

    def _log(self, msg):
        print(f"[{datetime.now():%H:%M:%S.%f}"[:-3] + f"] {self.dag_id}: {msg}")

    def _execute(self, t: Task, ctx: dict, results: dict):
        kwargs = {u: results[u] for u in t.upstream}          # upstream results as arguments
        for attempt in range(t.retries + 1):
            try:
                self._log(f"{t.name} attempt {attempt + 1}/{t.retries + 1}")
                results[t.name] = t.fn(ctx, **kwargs)
                return
            except Exception as e:
                self._log(f"{t.name} error: {e!r}")
                if attempt == t.retries:
                    raise
                time.sleep(t.retry_delay * 2 ** attempt)      # exponential backoff

    def run(self, ctx: dict | None = None, max_workers: int = 4) -> dict:
        ctx = ctx or {}
        graph = {n: set(t.upstream) for n, t in self.tasks.items()}
        sorter = TopologicalSorter(graph)
        sorter.prepare()                                      # raises CycleError if not a DAG
        state = {n: State.PENDING for n in self.tasks}
        results, running = {}, {}

        with ThreadPoolExecutor(max_workers) as pool:
            while sorter.is_active():
                for name in sorter.get_ready():               # tasks whose upstreams are finished
                    t = self.tasks[name]
                    if any(state[u] != State.SUCCESS for u in t.upstream):
                        state[name] = State.UPSTREAM_FAILED
                        self._log(f"{name} -> upstream_failed")
                        sorter.done(name)
                        continue
                    state[name] = State.RUNNING
                    running[pool.submit(self._execute, t, ctx, results)] = name
                if not running:
                    continue
                finished, _ = wait(running, return_when=FIRST_COMPLETED)
                for f in finished:
                    name = running.pop(f)
                    state[name] = State.FAILED if f.exception() else State.SUCCESS
                    self._log(f"{name} -> {state[name].value}")
                    sorter.done(name)
        return state

### Use it: a fan-in pipeline with a flaky source
```
extract_eu ─┐
extract_us ─┼─> transform ─> load
extract_asia┘
```

In [ ]:
attempts = {}                                  # to make failures deterministic

dag = MiniDAG("sales_etl")

@dag.task(retries=3)
def extract_eu(ctx):
    time.sleep(0.3); return [1, 2, 3]

@dag.task(retries=3)
def extract_us(ctx):
    attempts["us"] = attempts.get("us", 0) + 1
    if attempts["us"] < 3:                     # fails twice, then succeeds -> retries save the day
        raise ConnectionError("API timeout")
    time.sleep(0.3); return [4, 5]

@dag.task(retries=0)
def extract_asia(ctx):
    time.sleep(0.3); return [6]

@dag.task(after=[extract_eu, extract_us, extract_asia])
def transform(ctx, extract_eu, extract_us, extract_asia):
    return [x * 10 for x in extract_eu + extract_us + extract_asia]

@dag.task(after=[transform])
def load(ctx, transform):
    print("   >>> loading", transform, "for", ctx.get("ds"))
    return len(transform)

final_state = dag.run({"ds": "2025-01-01"})
print({k: v.value for k, v in final_state.items()})

**Observe:** the three extracts ran **in parallel**, `extract_us` was retried with backoff, `load` waited for everything.

### Failure propagation
Now make `extract_asia` fail permanently.

In [ ]:
dag2 = MiniDAG("broken_etl")

@dag2.task(retries=1, retry_delay=0.1)
def extract_a(ctx): raise RuntimeError("source down")

@dag2.task()
def extract_b(ctx): return [1]

@dag2.task(after=[extract_a, extract_b])
def transform(ctx, extract_a, extract_b): return "never runs"

@dag2.task(after=[transform])
def load(ctx, transform): return "never runs"

print({k: v.value for k, v in dag2.run().items()})

### Cycles are rejected
A graph with a cycle has no valid execution order – that's why it's a **D**AG.

In [ ]:
from graphlib import CycleError
bad = MiniDAG("cyclic")
a = Task("a", lambda c: 1, upstream=["b"]); b = Task("b", lambda c: 1, upstream=["a"])
bad.tasks = {"a": a, "b": b}
try:
    bad.run()
except CycleError as e:
    print("Rejected:", e.args[1])

**Quick question:** what is still missing compared with Airflow? *(persistent state/DB, scheduling by time, distributed workers, UI, templating, sensors, backfill CLI, pools, ...)*

## Part C – Idempotency & backfill
Two implementations of the same `load` task writing a daily partition to disk.

In [ ]:
import json, shutil
from pathlib import Path
from datetime import date, timedelta

OUT = Path("warehouse"); shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()

def fake_data_for(ds: str):                    # deterministic data per logical date
    n = int(ds[-2:])
    return [{"day": ds, "amount": 100 + n}]

def load_bad(ds):                              # ❌ append + depends on call count
    with open(OUT / "sales.jsonl", "a") as f:
        for r in fake_data_for(ds): f.write(json.dumps(r) + "\n")

def load_good(ds):                             # ✅ overwrite one partition atomically
    part = OUT / f"ds={ds}"; part.mkdir(exist_ok=True)
    tmp = part / "data.json.tmp"
    tmp.write_text(json.dumps(fake_data_for(ds)))
    tmp.replace(part / "data.json")            # atomic rename

for _ in range(3):                             # simulate 2 retries / manual reruns
    load_bad("2025-01-01"); load_good("2025-01-01")

print("bad  :", len((OUT / "sales.jsonl").read_text().splitlines()), "rows for one day (duplicates!)")
print("good :", len(json.loads((OUT / "ds=2025-01-01" / "data.json").read_text())), "row for one day")

### Backfill = run the same DAG for a range of logical dates
Possible *because* tasks are parametrized by `ds` and idempotent.

In [ ]:
def daterange(start: date, end: date):
    d = start
    while d <= end:
        yield d.isoformat(); d += timedelta(days=1)

for ds in daterange(date(2025, 1, 1), date(2025, 1, 5)):
    load_good(ds)

print(sorted(p.name for p in OUT.glob("ds=*")))

**Exercise (2 min):** add a `--force/--skip-if-exists` option to `load_good`. When is skipping safe? *(only if the partition is complete and inputs are immutable)*

## Part D – A real Airflow DAG
### D.1 (optional) Install Airflow
Skip if already installed. Uses the **official constraint file** (always do this for reproducibility). Takes a few minutes. You can use a newer 3.x version than the one set below.

In [ ]:
import importlib.util, sys, subprocess

AIRFLOW_VERSION = "3.0.6"     # change if needed
if importlib.util.find_spec("airflow") is None:
    py = f"{sys.version_info.major}.{sys.version_info.minor}"
    url = f"https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/constraints-{py}.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"apache-airflow=={AIRFLOW_VERSION}", "--constraint", url], check=True)
import airflow
print("Airflow", airflow.__version__)

### D.2 Write the DAG file
Features used: TaskFlow API, **dynamic task mapping** (`.expand`), retries with backoff, a `trigger_rule` notification task, templated logical date (`ds` injected as argument), idempotent output.

> Notice: **no heavy code at module top level** and **no `datetime.now()`**.

In [ ]:
from pathlib import Path
import os

AIRFLOW_HOME = Path("airflow_home").resolve()
DAGS = AIRFLOW_HOME / "dags"; DAGS.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile airflow_home/dags/sales_etl.py
from __future__ import annotations

import json
import random
from datetime import timedelta
from pathlib import Path

import pendulum

try:
    from airflow.sdk import dag, task            # Airflow 3
except ImportError:                              # Airflow 2.x
    from airflow.decorators import dag, task

OUT = Path("/tmp/airflow_course_out")


@dag(
    dag_id="sales_etl",
    schedule="@daily",
    start_date=pendulum.datetime(2025, 1, 1, tz="UTC"),
    catchup=False,
    max_active_runs=1,
    default_args={
        "retries": 2,
        "retry_delay": timedelta(seconds=2),
        "retry_exponential_backoff": True,
    },
    tags=["course", "etl"],
)
def sales_etl():

    @task
    def list_regions() -> list[str]:
        return ["eu", "us", "apac"]

    @task
    def extract(region: str, ds=None) -> dict:
        # pretend to call an API for the logical date `ds`
        rows = random.Random(f"{region}{ds}").randint(100, 200)
        return {"region": region, "rows": rows}

    @task
    def transform(batch: dict) -> dict:
        batch["rows_clean"] = int(batch["rows"] * 0.95)
        return batch

    @task
    def load(batches: list[dict], ds=None) -> str:
        # idempotent: one partition per logical date, overwritten on rerun
        part = OUT / f"ds={ds}"
        part.mkdir(parents=True, exist_ok=True)
        tmp = part / "data.json.tmp"
        tmp.write_text(json.dumps(list(batches)))
        tmp.replace(part / "data.json")
        return str(part / "data.json")             # pass a REFERENCE, not the data

    @task(trigger_rule="all_done")                 # runs even if something failed
    def notify(path: str | None = None):
        print(f"Pipeline finished. Output: {path}")

    regions = list_regions()
    extracted = extract.expand(region=regions)          # fan-out: 1 task instance per region
    transformed = transform.expand(batch=extracted)     # mapped -> mapped
    path = load(transformed)                            # fan-in: receives the list of results
    notify(path)


sales_etl()

### D.3 Parse it (what CI should always do)
`DagBag` loads the files exactly like the DAG processor does. Import errors and cycles show up here.

In [ ]:
env = {**os.environ,
       "AIRFLOW_HOME": str(AIRFLOW_HOME),
       "AIRFLOW__CORE__DAGS_FOLDER": str(DAGS),
       "AIRFLOW__CORE__LOAD_EXAMPLES": "False"}
os.environ.update(env)

try:
    from airflow.models import DagBag
    bag = DagBag(dag_folder=str(DAGS), include_examples=False)
    print("Import errors:", bag.import_errors or "none")
    d = bag.get_dag("sales_etl")
    print("Tasks:", [t.task_id for t in d.tasks])
    print("Schedule:", d.schedule if hasattr(d, "schedule") else d.timetable)
except ImportError:
    print("Airflow not installed -> run the install cell above (D.1).")

### D.4 Run it end-to-end without a scheduler
`airflow dags test` executes one DAG run in-process for a given logical date (ideal for local debugging).

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-2500:])

sh("airflow db migrate")
sh("airflow dags test sales_etl 2025-01-03")

In [ ]:
sh("find /tmp/airflow_course_out -type f | sort; cat /tmp/airflow_course_out/ds=2025-01-03/data.json")

### D.5 Backfill & inspect (CLI cheat-sheet)
```bash
airflow dags list
airflow dags backfill create --dag-id sales_etl --from-date 2025-01-01 --to-date 2025-01-05   # Airflow 3
airflow dags backfill -s 2025-01-01 -e 2025-01-05 sales_etl                                   # Airflow 2
airflow tasks test sales_etl extract 2025-01-03                       # run ONE task
airflow dags trigger sales_etl --logical-date 2025-01-04
airflow standalone                                                    # dev stack: scheduler + UI
```
Then open the UI (default <http://localhost:8080>) and look at **Grid view**, **Graph view**, **logs**, and the **mapped task instances**.

## Part E – Same idea, other tools (read-only, for discussion)
**Prefect** – plain Python, runs locally with no server:
```python
from prefect import flow, task

@task(retries=3, retry_delay_seconds=[1, 5, 20])
def extract(region: str) -> int: ...

@task
def load(counts: list[int]) -> int: return sum(counts)

@flow(log_prints=True)
def etl(regions=("eu", "us", "apac")):
    futures = extract.map(regions)                 # dynamic fan-out
    return load([f.result() for f in futures])

if __name__ == "__main__":
    etl()
```
**Dagster** – you declare *assets* (data), dependencies are inferred from function arguments:
```python
from dagster import asset, Definitions, DailyPartitionsDefinition

daily = DailyPartitionsDefinition(start_date="2025-01-01")

@asset(partitions_def=daily)
def raw_sales(context): ...

@asset(partitions_def=daily)
def clean_sales(raw_sales): ...

defs = Definitions(assets=[raw_sales, clean_sales])
```
| Question | Airflow | Prefect | Dagster |
|---|---|---|---|
| Unit of work | Task in a DAG | Task in a flow | Asset (data) |
| Dynamic graphs | `.expand()` | native Python loops / `.map()` | dynamic outputs / partitions |
| Strength | Ecosystem & maturity | Simplicity | Lineage & testing |

## Exercises
1. **Mini-orchestrator:** add a `timeout` per task (hint: `future.result(timeout=...)`) and a `trigger_rule="all_done"` option so a `notify` task runs even after failures.
2. **Mini-orchestrator:** add a `skipped` state — a branch task decides which downstream path runs.
3. **Airflow:** add a `BashOperator` (or `@task.bash`) that "compresses" the output, plus `execution_timeout=timedelta(minutes=5)` on `extract`.
4. **Airflow:** write a pytest test asserting no import errors, `len(dag.tasks) == 5`, and that `notify` has `trigger_rule == "all_done"`.
5. **Think:** you must process 10 000 files per day. Which Airflow features would you use, and what would you avoid (XCom payloads, one task per file at huge scale...)? What would you delegate to Spark?
6. **Compare:** reimplement the DAG in Prefect *or* Dagster and list 3 differences in developer experience.

### Clean-up

In [ ]:
shutil.rmtree("warehouse", ignore_errors=True)
shutil.rmtree("/tmp/airflow_course_out", ignore_errors=True)
# shutil.rmtree("airflow_home", ignore_errors=True)   # uncomment to remove the Airflow sandbox
print("done")